In [28]:
import pandas as pd
import os
from pathlib import Path


In [78]:
subpool_map = {
    'Subpool_2': '67A',
    'Subpool_3': '67B',
    'Subpool_4': '67C',
    'Subpool_5': '67D',
    'Subpool_6': '67E',
    'Subpool_7': '67F',
    'Subpool_8': '67G',
    'Subpool_9': '67H',
    'Subpool_10': '67I',
    'Subpool_11': '67J',
    'Subpool_12': '67K',
    'Subpool_13': '67L',
    'Subpool_14': '67M',
    'Subpool_15': '67N',
    'Subpool_16': '67O'
}


reversed_map = {
    '67A': 'Subpool_2',
    '67B': 'Subpool_3',
    '67C': 'Subpool_4',
    '67D': 'Subpool_5',
    '67E': 'Subpool_6',
    '67F': 'Subpool_7',
    '67G': 'Subpool_8',
    '67H': 'Subpool_9',
    '67I': 'Subpool_10',
    '67J': 'Subpool_11',
    '67K': 'Subpool_12',
    '67L': 'Subpool_13',
    '67M': 'Subpool_14',
    '67N': 'Subpool_15',
    '67O': 'Subpool_16'
}


In [19]:
example_config = pd.read_csv('/share/crsp/lab/seyedam/weberrl/modified_parse_pipeline/configs/igvf_b01_config.tsv', sep = '\t')

In [135]:
example_config

,fastq,fastq_r2,subpool,sample,plate,lane,run,platform,droplets_included,learning_rate,expected_cells,igvf_adata
0,/dfs7/samlab/seyedam/IGVF/igvf_b01/next1/B01_1...,/dfs7/samlab/seyedam/IGVF/igvf_b01/next1/B01_1...,Subpool_8,S8,igvf_b01,L001,1,next,40000,0.0001,5000,/share/crsp/lab/seyedam/share/igvf_bridge/data...
1,/dfs7/samlab/seyedam/IGVF/igvf_b01/next2/B01_1...,/dfs7/samlab/seyedam/IGVF/igvf_b01/next2/B01_1...,Subpool_8,S8,igvf_b01,L001,2,next,40000,0.0001,5000,/share/crsp/lab/seyedam/share/igvf_bridge/data...
2,/dfs7/samlab/seyedam/IGVF/igvf_b01/next1/B01_1...,/dfs7/samlab/seyedam/IGVF/igvf_b01/next1/B01_1...,Subpool_1,S1,igvf_b01,L001,1,next,40000,0.0001,5000,/share/crsp/lab/seyedam/share/igvf_bridge/data...
3,/dfs7/samlab/seyedam/IGVF/igvf_b01/next2/B01_1...,/dfs7/samlab/seyedam/IGVF/igvf_b01/next2/B01_1...,Subpool_1,S1,igvf_b01,L001,1,next,40000,0.0001,5000,/share/crsp/lab/seyedam/share/igvf_bridge/data...


In [6]:
def find_config_tsv_files(main_dir):
    return [
        os.path.join(root, file)
        for root, dirs, files in os.walk(main_dir)
        for file in files
        if file.endswith('_config.tsv')
    ]



In [95]:
configs = find_config_tsv_files('/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old')
configs_dict = {
    os.path.basename(path).replace('_config.tsv', ''): path
    for path in configs
}

exclude_keys = {'igvf_b01', 'igvf_024a', 'igvf_015', 'igvf_014', 'igvf_013'}

configs_dict = {k: v for k, v in configs_dict.items() if k not in exclude_keys}



In [96]:
configs_dict

{'igvf_007': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_007_config.tsv',
 'igvf_021': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_021_config.tsv',
 'igvf_008': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_008_config.tsv',
 'igvf_020': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_020_config.tsv',
 'igvf_004': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_004_config.tsv',
 'igvf_019': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_019_config.tsv',
 'igvf_009': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_009_config.tsv',
 'igvf_022': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_022_config.tsv',
 'igvf_016': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_016_config.tsv',
 'igvf_023': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/conf

In [111]:
error_configs = {}  # Store configs that failed

for name, path in configs_dict.items():
    if not path.endswith('.tsv'):
        print(f"⚠️ Skipping file (not .tsv extension): {path}")
        error_configs[name] = path
        continue

    df = read_config_file(path)
    if df is None:
        error_configs[name] = path
        continue

    if 'subpool' not in df.columns:
        print(f"❌ Missing 'subpool' column in {path}")
        error_configs[name] = path
        continue

    def normalize_subpool(sp):
        if sp in subpool_map:
            return subpool_map[sp]
        elif sp in subpool_map.values():
            return sp
        else:
            raise ValueError(f"Unknown subpool '{sp}' in {path}")

    try:
        df['subpool'] = df['subpool'].apply(normalize_subpool)
    except Exception as e:
        print(f"❌ {e}")
        error_configs[name] = path
        continue

    for col, val in defaults.items():
        if col not in df.columns:
            df[col] = val

    output_path = os.path.join(output_dir, os.path.basename(path))
    df = df.reset_index(drop=True)
    df.to_csv(output_path, sep='\t', index=False)
    print(f"✅ Saved cleaned config: {output_path}")


✅ Saved cleaned config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_007_config.tsv
❌ Unknown subpool 'Subpool_UDI17' in /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_021_config.tsv
✅ Saved cleaned config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_008_config.tsv
❌ Unknown subpool 'Subpool_UDI01' in /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_020_config.tsv
✅ Saved cleaned config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_004_config.tsv
❌ Unknown subpool 'Subpool_UDI41' in /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_019_config.tsv
✅ Saved cleaned config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_009_config.tsv
✅ Saved cleaned config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_022_config.ts

In [115]:
igvf_012_map = {'Subpool_1':'13A',
 'Subpool_2':'13B',
 'Subpool_3':'13C',
 'Subpool_4':'13D',
 'Subpool_5':'13E',
 'Subpool_6':'13F',
 'Subpool_7':'13G',
 'Subpool_8':'13H'} 

igvf_023_map = {
    'S1':'67A',
    'S2':'67B',
    'S3':'67C',
    'S4':'67D',
    'S5':'67E',
    'S6':'67F',
    'S7':'67G',
    'S8':'67H',
    'S9':'67I',
    'S10':'67J',
    'S11':'67K',
    'S12':'67L',
    'S13':'67M',
    'S14':'67N',
    'S15':'67O'    
}

In [99]:
error_configs

{'igvf_021': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_021_config.tsv',
 'igvf_020': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_020_config.tsv',
 'igvf_019': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_019_config.tsv',
 'igvf_023': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_023_config.tsv',
 'igvf_012': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_012_config.tsv'}

In [100]:
mapping_text = """
igvf_021	021_67A	UDI17
igvf_021	021_67B	UDI18
igvf_021	021_67C	UDI19
igvf_021	021_67D	UDI20
igvf_021	021_67E	UDI21
igvf_021	021_67F	UDI22
igvf_021	021_67G	UDI23
igvf_021	021_67H	UDI24
igvf_021	021_67I	UDI25
igvf_021	021_67J	UDI26
igvf_021	021_67K	UDI27
igvf_021	021_67L	UDI28
igvf_021	021_67M	UDI29
igvf_021	021_67N	UDI30
igvf_021	021_67O	UDI31
igvf_020	020_67A	UDI01
igvf_020	020_67B	UDI02
igvf_020	020_67C	UDI03
igvf_020	020_67D	UDI04
igvf_020	020_67E	UDI05
igvf_020	020_67F	UDI06
igvf_020	020_67G	UDI07
igvf_020	020_67H	UDI08
igvf_020	020_67I	UDI09
igvf_020	020_67J	UDI10
igvf_020	020_67K	UDI11
igvf_020	020_67L	UDI12
igvf_020	020_67M	UDI13
igvf_020	020_67N	UDI14
igvf_020	020_67O	UDI15
igvf_019	019_67A	UDI41
igvf_019	019_67B	UDI42
igvf_019	019_67C	UDI43
igvf_019	019_67D	UDI44
igvf_019	019_67E	UDI45
igvf_019	019_67F	UDI46
igvf_019	019_67G	UDI47
igvf_019	019_67H	UDI48
igvf_019	019_67I	UDI20
igvf_019	019_67J	UDI21
igvf_019	019_67K	UDI22
igvf_019	019_67L	UDI23
igvf_019	019_67M	UDI24
igvf_019	019_67N	UDI29
igvf_019	019_67O	UDI30
"""


In [101]:
from collections import defaultdict

# Initialize nested dictionary
udi_to_subpool_map = defaultdict(dict)

# Parse each line
for line in mapping_text.strip().splitlines():
    dataset, subpool_code, udi_id = line.strip().split('\t')
    subpool = subpool_code.split('_')[1]  # extract '67A' from '021_67A'
    udi_key = f"Subpool_{udi_id}"
    udi_to_subpool_map[dataset][udi_key] = subpool

# Convert defaultdict to normal dict if needed
udi_to_subpool_map = dict(udi_to_subpool_map)


In [108]:
def read_config_file(path):
    try:
        df = pd.read_csv(path, sep='\t')
        if df.shape[1] == 1:
            df = pd.read_csv(path, sep=',')
    except Exception as e:
        print(f"❌ Failed to read {path}: {e}")
        return None
    return df

# Only attempt to fix the three datasets we now have mappings for
for name, path in error_configs.items():
    if name not in udi_to_subpool_map:
        print(f"⏭ Skipping {name}, no UDI mapping available.")
        continue

    df = read_config_file(path)
    if df is None or 'subpool' not in df.columns:
        print(f"❌ Failed to read or missing 'subpool' in {name}")
        continue

    # Apply UDI normalization
    try:
        df['subpool'] = df['subpool'].apply(lambda sp: udi_to_subpool_map[name][sp])
    except KeyError as e:
        print(f"❌ Unknown UDI subpool {e} in {name}")
        continue

    # Fill missing columns
    for col, val in defaults.items():
        if col not in df.columns:
            df[col] = val

    # Save corrected file
    output_path = os.path.join(output_dir, os.path.basename(path))
    df.to_csv(output_path, sep='\t', index=False)
    print(f"✅ Fixed UDI-based config: {output_path}")


✅ Fixed UDI-based config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_021_config.tsv
✅ Fixed UDI-based config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_020_config.tsv
✅ Fixed UDI-based config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_019_config.tsv
⏭ Skipping igvf_023, no UDI mapping available.
⏭ Skipping igvf_012, no UDI mapping available.


In [116]:
error_configs.

{'igvf_021': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_021_config.tsv',
 'igvf_020': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_020_config.tsv',
 'igvf_019': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_019_config.tsv',
 'igvf_023': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_023_config.tsv',
 'igvf_012': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_012_config.tsv'}

In [ ]:
igvf_012_map = {'Subpool_1':'13A',
 'Subpool_2':'13B',
 'Subpool_3':'13C',
 'Subpool_4':'13D',
 'Subpool_5':'13E',
 'Subpool_6':'13F',
 'Subpool_7':'13G',
 'Subpool_8':'13H'} 

igvf_023_map = {
    'S1':'67A',
    'S2':'67B',
    'S3':'67C',
    'S4':'67D',
    'S5':'67E',
    'S6':'67F',
    'S7':'67G',
    'S8':'67H',
    'S9':'67I',
    'S10':'67J',
    'S11':'67K',
    'S12':'67L',
    'S13':'67M',
    'S14':'67N',
    'S15':'67O'    
}

In [121]:
exclude_keys = ['igvf_021','igvf_020','igvf_019']
configs_dict_final = {k: v for k, v in error_configs.items() if k not in exclude_keys}


In [120]:
configs_dict

{'igvf_023': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_023_config.tsv',
 'igvf_012': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_012_config.tsv'}

In [122]:
import os
import pandas as pd

# Input config dictionary
configs_dict_final = {
    'igvf_023': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_023_config.tsv',
    'igvf_012': '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_012_config.tsv'
}

# Subpool mappings
igvf_012_map = {
    'Subpool_1': '13A', 'Subpool_2': '13B', 'Subpool_3': '13C', 'Subpool_4': '13D',
    'Subpool_5': '13E', 'Subpool_6': '13F', 'Subpool_7': '13G', 'Subpool_8': '13H'
}

igvf_023_map = {
    'S1': '67A', 'S2': '67B', 'S3': '67C', 'S4': '67D', 'S5': '67E',
    'S6': '67F', 'S7': '67G', 'S8': '67H', 'S9': '67I', 'S10': '67J',
    'S11': '67K', 'S12': '67L', 'S13': '67M', 'S14': '67N', 'S15': '67O'
}

subpool_maps = {
    'igvf_012': igvf_012_map,
    'igvf_023': igvf_023_map
}

defaults = {
    'droplets_included': 200000,
    'learning_rate': 0.00005,
    'expected_cells': 67000
}

output_dir = '/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp'
os.makedirs(output_dir, exist_ok=True)

def read_config_file(path):
    try:
        df = pd.read_csv(path, sep='\t')
        if df.shape[1] == 1:
            df = pd.read_csv(path, sep=',')
    except Exception as e:
        print(f"❌ Failed to read {path}: {e}")
        return None
    return df

for name, path in configs_dict_final.items():
    df = read_config_file(path)
    if df is None or 'subpool' not in df.columns:
        print(f"❌ Failed to read or missing 'subpool' in {name}")
        continue

    subpool_map = subpool_maps.get(name)
    if not subpool_map:
        print(f"⏭ No mapping available for {name}")
        continue

    # Normalize subpool
    try:
        df['subpool'] = df['subpool'].apply(lambda sp: subpool_map[sp])
    except KeyError as e:
        print(f"❌ Unknown subpool {e} in {name}")
        continue

    # Fill missing columns
    for col, val in defaults.items():
        if col not in df.columns:
            df[col] = val

    # Drop Unnamed: 0 if it exists
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns='Unnamed: 0')

    df = df.reset_index(drop=True)

    # Save cleaned file
    output_path = os.path.join(output_dir, os.path.basename(path))
    df.to_csv(output_path, sep='\t', index=False)
    print(f"✅ Fixed mapped config: {output_path}")


✅ Fixed mapped config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_023_config.tsv
✅ Fixed mapped config: /share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs/configs_fixed_temp/igvf_012_config.tsv


In [84]:
configs_dict['igvf_003']

'/share/crsp/lab/seyedam/share/parse_pipeline_from_igvf/configs_old/igvf_003_config.tsv'

In [87]:
old_config = pd.read_csv(configs_dict['igvf_003'], sep = ',')

In [88]:
old_config

,Unnamed: 0,fastq,fastq_r2,subpool,sample,plate,lane,run,platform,droplets_included,learning_rate,expected_cells
0,0,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,Subpool_10,S9,igvf_003,L001,1,nova,200000,0.00005,67000
1,1,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,Subpool_10,S9,igvf_003,L002,1,nova,200000,0.00005,67000
2,2,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,Subpool_10,S9,igvf_003,L003,1,nova,200000,0.00005,67000
3,3,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,Subpool_10,S9,igvf_003,L004,1,nova,200000,0.00005,67000
4,4,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,Subpool_10,S9,igvf_003,L001,2,nova,200000,0.00005,67000
...,...,...,...,...,...,...,...,...,...,...,...,...
115,115,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Subli...,Subpool_9,S8,igvf_003,L004,1,nova,200000,0.00005,67000
116,116,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,Subpool_9,S8,igvf_003,L001,2,nova,200000,0.00005,67000
117,117,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,Subpool_9,S8,igvf_003,L002,2,nova,200000,0.00005,67000
118,118,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,/dfs7/samlab/seyedam/IGVF/igvf_003/nova2/Subli...,Subpool_9,S8,igvf_003,L003,2,nova,200000,0.00005,67000


In [133]:
igvf_017 = pd.read_csv('configs_fixed_temp/igvf_017_config.tsv', sep = '\t')

In [134]:
igvf_017

,Unnamed: 0.1,Unnamed: 0,fastq,fastq_r2,subpool,plate,lane,run,platform,droplets_included,learning_rate,expected_cells
0,0,0,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,67A,igvf_017,L1,1,nova,200000,0.00005,44000
1,1,1,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,67A,igvf_017,L1,1,nova,200000,0.00005,44000
2,2,2,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,67O,igvf_017,L1,1,nova,200000,0.00005,44000
3,3,3,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq/017...,67O,igvf_017,L1,1,nova,200000,0.00005,44000
4,4,4,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,67B,igvf_017,L1,1,nova,200000,0.00005,44000
5,5,5,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,67C,igvf_017,L1,1,nova,200000,0.00005,44000
6,6,6,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,67D,igvf_017,L1,1,nova,200000,0.00005,44000
7,7,7,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,67E,igvf_017,L1,1,nova,200000,0.00005,44000
8,8,8,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,67F,igvf_017,L1,1,nova,200000,0.00005,44000
9,9,9,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,/dfs7/samlab/seyedam/IGVF/igvf_017/nextseq2/01...,67G,igvf_017,L1,1,nova,200000,0.00005,44000


In [26]:
h5ad_path = '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003'

In [35]:

# Convert string to Path object
h5ad_path = Path('/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003')

# Recursively list all files
file_list_str = [str(f) for f in file_list]


In [42]:
file_list_str

['/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67M.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67E.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67G.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67I.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67J.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67D.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67A.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67C.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_67K.h5ad',
 '/share/crsp/lab/seyedam/weberrl/igvf_portal_data/scripts/h5ad_from_portal/igvf_003/003_13A.h5ad',


In [43]:
old_config['fastq'][0]

'/dfs7/samlab/seyedam/IGVF/igvf_003/nova1/Sublibrary_10_S9_L001_R1_001.fastq.gz'

In [74]:
old_config['subpool'].unique()

array(['Subpool_10', 'Subpool_11', 'Subpool_12', 'Subpool_13',
       'Subpool_14', 'Subpool_15', 'Subpool_16', 'Subpool_2', 'Subpool_3',
       'Subpool_4', 'Subpool_5', 'Subpool_6', 'Subpool_7', 'Subpool_8',
       'Subpool_9'], dtype=object)